# Catcher P4 or Not Models — Train V2

Split the **hitter** CSV into just the catchers to do model development.

Building out the P4 or not models for catchers. Trained on D1 catchers only (P4 + Non P4 D1).

Uses 8 production features (including `c_velo` and `pop_time`) plus `d1_prob` as a meta-feature.

**Source:** `backend/ml/train/train_v2/hitters/hitters_cleaned.csv`

In [11]:
import pandas as pd
import numpy as np
import re
import os
from pathlib import Path

HITTER_CSV = Path("hitters_cleaned.csv")

hitters = pd.read_csv(HITTER_CSV, low_memory=False)
print(f"Loaded {len(hitters):,} rows, {len(hitters.columns)} columns")

catchers = hitters[hitters["resolved_position"] == "C"]
catchers.head()

Loaded 32,999 rows, 50 columns


,name,link,player_state,high_school,class,resolved_position,positions,commitment,commitment_date,height,...,of_velo_date,c_velo,c_velo_date,pop_time,pop_time_date,player_region,conference,division,college_location,committment_group
11,Graydon Ambrose,https://www.prepbaseballreport.com/profiles/VA...,VA,First Colonial,2027,C,"C,1B",Virginia Tech,10/16/25,73.0,...,NaN,80.0,2/22/26,1.92,7/23/25,South,Atlantic Coast Conference,NCAA I,"BLACKSBURG, VA",P4
14,KJ Anderson,https://www.prepbaseballreport.com/profiles/TN...,TN,Cookeville,2027,C,"C,3B",Auburn,9/08/25,72.0,...,NaN,82.0,5/27/25,1.84,5/27/25,South,Southeastern Conference,NCAA I,"AUBURN, AL",P4
26,Connor Balazic,https://www.prepbaseballreport.com/profiles/FL...,FL,Creekside,2027,C,"C,SS",North Carolina (9/08/25),NaN,70.0,...,NaN,84.0,6/04/25,1.90,9/13/25,South,Atlantic Coast Conference,NCAA I,"CHAPEL HILL, NC",P4
28,Mason Ballard,https://www.prepbaseballreport.com/profiles/MS...,MS,Tupelo,2027,C,"C,3B",Mississippi Gulf Coast Jc,11/09/25,70.0,...,NaN,82.0,7/23/25,1.89,7/23/25,South,NaN,NJCAA II,"PERKINSTON, MS",Non D1
29,Jayden Barker,https://www.prepbaseballreport.com/profiles/LA...,LA,Slidell,2027,C,"C,1B",John Melvin University,NaN,71.0,...,NaN,NaN,NaN,NaN,NaN,South,NaN,USCAA,"CROWLEY, LA",Non D1


In [12]:
C_MODELING_COLS = [
    "player_state", "resolved_position",
    "height", "weight", "throwing_hand", "hitting_handedness",
    "exit_velo_max", "exit_velo_max_date",
    "exit_velo_avg", "exit_velo_avg_date",
    "distance_max", "distance_max_date",
    "sweet_spot_p", "sweet_spot_p_date",
    "hand_speed_max", "hand_speed_max_date",
    "bat_speed_max", "bat_speed_max_date",
    "rot_acc_max", "rot_acc_max_date",
    "hard_hit_p", "hard_hit_p_date",
    "sixty_time", "sixty_time_date",
    "thirty_time", "thirty_time_date",
    "ten_yard_time", "ten_yard_time_date",
    "run_speed_max", "run_speed_max_date",
    "c_velo", "c_velo_date",
    "pop_time", "pop_time_date",
    "player_region", "committment_group", "commitment_date"
]

C_MODELING_COLS = [c for c in C_MODELING_COLS if c in catchers.columns]
c_modeling = catchers[C_MODELING_COLS]

c_modeling = c_modeling[c_modeling["committment_group"] != "Unknown"]
c_modeling = c_modeling[c_modeling["committment_group"] != "Non D1"]

print(c_modeling["committment_group"].value_counts())
print(c_modeling.shape)

committment_group
Non P4 D1    1226
P4            277
Name: count, dtype: int64
(1503, 37)


In [13]:
c_modeling["p4_or_not"] = (c_modeling["committment_group"] != "Non P4 D1").astype(int)

c_modeling["p4_or_not"].value_counts()

p4_or_not
0    1226
1     277
Name: count, dtype: int64

In [14]:
# ============================================================
# STALE DATA ANALYSIS
# ============================================================
STALE_MONTHS = 24
OUTLIER_STD_DEV = 2

c_model_recent = c_modeling.copy()
c_model_recent["class"] = catchers.loc[c_model_recent.index, "class"]

STAT_DATE_PAIRS = [
    ("exit_velo_max",  "exit_velo_max_date"),
    ("exit_velo_avg",  "exit_velo_avg_date"),
    ("distance_max",   "distance_max_date"),
    ("sweet_spot_p",   "sweet_spot_p_date"),
    ("hand_speed_max", "hand_speed_max_date"),
    ("bat_speed_max",  "bat_speed_max_date"),
    ("rot_acc_max",    "rot_acc_max_date"),
    ("hard_hit_p",     "hard_hit_p_date"),
    ("sixty_time",     "sixty_time_date"),
    ("thirty_time",    "thirty_time_date"),
    ("ten_yard_time",  "ten_yard_time_date"),
    ("run_speed_max",  "run_speed_max_date"),
    ("c_velo",         "c_velo_date"),
    ("pop_time",       "pop_time_date"),
]
STAT_DATE_PAIRS = [(s, d) for s, d in STAT_DATE_PAIRS if s in c_model_recent.columns and d in c_model_recent.columns]

def parse_pbr_date(d):
    if pd.isna(d) or str(d).strip() == "":
        return pd.NaT
    try:
        return pd.to_datetime(str(d).strip(), format="%m/%d/%y")
    except Exception:
        try:
            return pd.to_datetime(str(d).strip())
        except Exception:
            return pd.NaT

c_model_recent["grad_date"] = pd.to_datetime(c_model_recent["class"].astype(str) + "-06-01")

for stat_col, date_col in STAT_DATE_PAIRS:
    parsed_col = f"_{date_col}_parsed"
    months_col = f"{stat_col}__months_before_grad"
    c_model_recent[parsed_col] = c_model_recent[date_col].apply(parse_pbr_date)
    c_model_recent[months_col] = ((c_model_recent["grad_date"] - c_model_recent[parsed_col]).dt.days / 30.44).round(1)

print(f"Parsed {len(STAT_DATE_PAIRS)} stat date columns.\n")

print(f"{'='*70}")
print(f"STALENESS BY COMMITMENT GROUP  (threshold = {STALE_MONTHS} months)")
print(f"{'='*70}\n")

for group in ["P4", "Non P4 D1"]:
    mask = c_model_recent["committment_group"] == group
    g_stale = 0
    g_total = 0
    for stat_col, _ in STAT_DATE_PAIRS:
        months_col = f"{stat_col}__months_before_grad"
        has_val = c_model_recent.loc[mask, stat_col].notna()
        is_stale = c_model_recent.loc[mask, months_col] > STALE_MONTHS
        g_stale += (has_val & is_stale).sum()
        g_total += has_val.sum()
    pct = 100 * g_stale / g_total if g_total else 0
    n_players = mask.sum()
    print(f"  {group:>12}  ({n_players:>5,} players): {g_stale:>4,} / {g_total:>5,} values stale ({pct:.1f}%)")

stat_value_cols = [s for s, _ in STAT_DATE_PAIRS]
print(f"\n{'='*70}")
print(f"OUTLIER SUMMARY  (+-{OUTLIER_STD_DEV} std dev from group mean)")
print(f"{'='*70}\n")

for group in ["P4", "Non P4 D1"]:
    mask = c_model_recent["committment_group"] == group
    tot_outliers = 0
    tot_stale_o = 0
    for stat_col in stat_value_cols:
        months_col = f"{stat_col}__months_before_grad"
        vals = c_model_recent.loc[mask, stat_col]
        mean = vals.mean()
        std = vals.std()
        if pd.isna(mean) or pd.isna(std) or std == 0:
            continue
        is_outlier = (vals < mean - OUTLIER_STD_DEV * std) | (vals > mean + OUTLIER_STD_DEV * std)
        is_stale = c_model_recent.loc[mask, months_col] > STALE_MONTHS
        tot_outliers += is_outlier.sum()
        tot_stale_o += (is_outlier & is_stale).sum()
    tot_fresh_o = tot_outliers - tot_stale_o
    pct = 100 * tot_stale_o / tot_outliers if tot_outliers else 0
    print(f"  {group:>12}: {tot_outliers:>4} outliers -> {tot_stale_o:>3} stale ({pct:.0f}%), {tot_fresh_o:>3} fresh ({100-pct:.0f}%)")

internal_cols = [c for c in c_model_recent.columns if c.startswith("_")]
c_model_recent.drop(columns=internal_cols, inplace=True)
print(f"\nc_model_recent shape: {c_model_recent.shape}")

Parsed 14 stat date columns.

STALENESS BY COMMITMENT GROUP  (threshold = 24 months)

            P4  (  277 players):  792 / 2,699 values stale (29.3%)
     Non P4 D1  (1,226 players): 3,051 / 11,680 values stale (26.1%)

OUTLIER SUMMARY  (+-2 std dev from group mean)

            P4:  118 outliers ->  74 stale (63%),  44 fresh (37%)
     Non P4 D1:  503 outliers -> 246 stale (49%), 257 fresh (51%)

c_model_recent shape: (1503, 54)


In [15]:
# ============================================================
# APPLY OPTION B — NaN only stale outliers (MULTI-PASS)
# ============================================================
MAX_PASSES = 5
grand_total = 0

for pass_i in range(1, MAX_PASSES + 1):
    pass_total = 0
    pass_log = []

    for stat_col, date_col in STAT_DATE_PAIRS:
        months_col = f"{stat_col}__months_before_grad"
        stat_nand = 0

        for group in ["P4", "Non P4 D1"]:
            mask = c_model_recent["committment_group"] == group
            vals = c_model_recent.loc[mask, stat_col]
            mean = vals.mean()
            std = vals.std()

            if pd.isna(mean) or pd.isna(std) or std == 0:
                continue

            is_outlier = (vals < mean - OUTLIER_STD_DEV * std) | (vals > mean + OUTLIER_STD_DEV * std)
            is_stale = c_model_recent.loc[mask, months_col] > STALE_MONTHS

            to_nan = is_outlier & is_stale
            n = to_nan.sum()

            if n > 0:
                c_model_recent.loc[to_nan[to_nan].index, stat_col] = np.nan
                c_model_recent.loc[to_nan[to_nan].index, date_col] = np.nan
                stat_nand += n

        if stat_nand > 0:
            pass_log.append({"stat": stat_col, "values_removed": stat_nand})
            pass_total += stat_nand

    grand_total += pass_total
    print(f"Pass {pass_i}: NaN'd {pass_total} stale outlier values")
    if pass_log:
        for row in pass_log:
            print(f"    {row['stat']:>15}: {row['values_removed']}")

    if pass_total == 0:
        print(f"  No more stale outliers found — stopping.")
        break

print(f"\nTotal across {pass_i} passes: {grand_total} values NaN'd")

remaining = 0
for stat_col, _ in STAT_DATE_PAIRS:
    months_col = f"{stat_col}__months_before_grad"
    for group in ["P4", "Non P4 D1"]:
        mask = c_model_recent["committment_group"] == group
        vals = c_model_recent.loc[mask, stat_col]
        mean = vals.mean()
        std = vals.std()
        if pd.isna(mean) or pd.isna(std) or std == 0:
            continue
        is_outlier = (vals < mean - OUTLIER_STD_DEV * std) | (vals > mean + OUTLIER_STD_DEV * std)
        is_stale = c_model_recent.loc[mask, months_col] > STALE_MONTHS
        remaining += (is_outlier & is_stale).sum()

print(f"Stale outliers remaining: {remaining}")
print(f"c_model_recent shape: {c_model_recent.shape}")

Pass 1: NaN'd 320 stale outlier values
      exit_velo_max: 41
      exit_velo_avg: 27
       distance_max: 29
       sweet_spot_p: 29
     hand_speed_max: 16
      bat_speed_max: 23
        rot_acc_max: 19
         hard_hit_p: 6
         sixty_time: 27
        thirty_time: 4
      ten_yard_time: 7
      run_speed_max: 11
             c_velo: 35
           pop_time: 46
Pass 2: NaN'd 114 stale outlier values
      exit_velo_max: 21
      exit_velo_avg: 9
       distance_max: 11
       sweet_spot_p: 4
     hand_speed_max: 4
      bat_speed_max: 12
        rot_acc_max: 6
         hard_hit_p: 3
         sixty_time: 11
        thirty_time: 3
      run_speed_max: 1
             c_velo: 14
           pop_time: 15
Pass 3: NaN'd 29 stale outlier values
      exit_velo_max: 9
      exit_velo_avg: 2
       distance_max: 2
       sweet_spot_p: 1
     hand_speed_max: 1
      bat_speed_max: 3
         sixty_time: 6
        thirty_time: 1
           pop_time: 4
Pass 4: NaN'd 5 stale outlier values
  

## Logistic Regression Baseline

In [16]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.impute import KNNImputer
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, confusion_matrix

FEATURES = [
    "height", "weight",
    "exit_velo_max", "distance_max", "bat_speed_max",
    "sixty_time",
    "c_velo", "pop_time",
]

X = c_model_recent[FEATURES]
y = c_model_recent["p4_or_not"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

pipe = Pipeline([
    ("impute", KNNImputer(n_neighbors=10)),
    ("scale", StandardScaler()),
    ("lr", LogisticRegression(max_iter=1000, class_weight="balanced", random_state=42)),
])

pipe.fit(X_train, y_train)
y_pred = pipe.predict(X_test)

print(classification_report(y_test, y_pred, target_names=["Non P4 D1", "P4"]))
print(confusion_matrix(y_test, y_pred), "\n")

coefs = pd.Series(pipe["lr"].coef_[0], index=FEATURES).sort_values()
print(coefs)

              precision    recall  f1-score   support

   Non P4 D1       0.91      0.62      0.74       246
          P4       0.30      0.73      0.43        55

    accuracy                           0.64       301
   macro avg       0.61      0.67      0.58       301
weighted avg       0.80      0.64      0.68       301

[[153  93]
 [ 15  40]] 

pop_time        -0.207618
sixty_time      -0.095403
distance_max    -0.050859
weight          -0.013508
bat_speed_max    0.020721
c_velo           0.060477
height           0.195131
exit_velo_max    0.424485
dtype: float64


## Generate Out-of-Fold D1 Probabilities

We need `d1_prob` as a meta-feature for the P4 model. Can't use the saved D1 model directly because these D1 players were in its training data — that would leak. Instead, retrain the D1 model on the **full catcher dataset** (including Non-D1) with cross_val_predict to get leak-free calibrated probabilities for every player, then filter to just the D1 subset.

In [17]:
from xgboost import XGBClassifier
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.calibration import CalibratedClassifierCV
import numpy as np

# Load the FULL catcher dataset (including Non-D1) for D1 model training
hitters_full = pd.read_csv(Path("hitters_cleaned.csv"), low_memory=False)
c_full = hitters_full[hitters_full["resolved_position"] == "C"].copy()
c_full = c_full[c_full["committment_group"] != "Unknown"]
c_full["d1_or_not"] = (c_full["committment_group"] != "Non D1").astype(int)

FEATURES_D1 = ["height", "weight", "exit_velo_max", "distance_max", "sixty_time", "c_velo", "pop_time", "bat_speed_max"]

X_full = c_full[FEATURES_D1]
y_full = c_full["d1_or_not"]
neg, pos = (y_full == 0).sum(), (y_full == 1).sum()

print(f"Full catcher dataset: {len(c_full)} players ({pos} D1, {neg} Non-D1)")

xgb_d1 = XGBClassifier(
    n_estimators=500, max_depth=4, learning_rate=0.01,
    subsample=0.8, colsample_bytree=0.8,
    reg_alpha=1.0, reg_lambda=3.0, min_child_weight=5,
    scale_pos_weight=neg / pos, eval_metric="logloss",
    random_state=42, enable_categorical=False,
)
cal_d1 = CalibratedClassifierCV(xgb_d1, method="isotonic", cv=3)

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
oof_probs = cross_val_predict(cal_d1, X_full, y_full, cv=skf, method="predict_proba")[:, 1]

c_full["d1_prob"] = oof_probs

# Filter to D1 players only and merge into c_model_recent
d1_players = c_full[c_full["d1_or_not"] == 1][["d1_prob"]]
c_model_recent = c_model_recent.join(d1_players, how="left")

print(f"\nD1 prob stats for D1 players (n={c_model_recent['d1_prob'].notna().sum()}):")
print(f"  mean: {c_model_recent['d1_prob'].mean():.3f}")
print(f"  std:  {c_model_recent['d1_prob'].std():.3f}")
print(f"  min:  {c_model_recent['d1_prob'].min():.3f}")
print(f"  max:  {c_model_recent['d1_prob'].max():.3f}")

print(f"\nD1 prob by group:")
for group in ["P4", "Non P4 D1"]:
    mask = c_model_recent["committment_group"] == group
    vals = c_model_recent.loc[mask, "d1_prob"]
    print(f"  {group:>12}: mean={vals.mean():.3f}  std={vals.std():.3f}  median={vals.median():.3f}")

Full catcher dataset: 6258 players (1503 D1, 4755 Non-D1)

D1 prob stats for D1 players (n=1503):
  mean: 0.371
  std:  0.203
  min:  0.000
  max:  1.000

D1 prob by group:
            P4: mean=0.451  std=0.226  median=0.427
     Non P4 D1: mean=0.353  std=0.193  median=0.351


## XGBoost — P4 vs Non-P4 D1 (8 core + d1_prob)

Using the 8 production features plus the D1 probability as a meta-feature.

In [18]:
from xgboost import XGBClassifier
from sklearn.model_selection import StratifiedKFold, cross_validate, train_test_split
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score
from sklearn.calibration import CalibratedClassifierCV, calibration_curve
from sklearn.metrics import brier_score_loss

FEATURES_P4 = ["height", "weight", "exit_velo_max", "distance_max", "sixty_time", "c_velo", "pop_time", "bat_speed_max", "d1_prob"]

X = c_model_recent[FEATURES_P4]
y = c_model_recent["p4_or_not"]
neg, pos = (y == 0).sum(), (y == 1).sum()
print(f"Class balance: {neg} Non-P4 D1, {pos} P4  (ratio {neg/pos:.2f}:1)\n")

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

xgb_p4 = XGBClassifier(
    n_estimators=300, max_depth=3, learning_rate=0.03,
    subsample=0.8, colsample_bytree=0.8,
    reg_alpha=1.5, reg_lambda=5.0, min_child_weight=8,
    scale_pos_weight=neg / pos, eval_metric="logloss",
    random_state=42, enable_categorical=False,
)

xgb_p4.fit(X_train, y_train)
y_pred = xgb_p4.predict(X_test)
y_proba = xgb_p4.predict_proba(X_test)[:, 1]

print(f"XGBoost — {len(FEATURES_P4)} Features (8 core + d1_prob)")
print("=" * 60)
print(classification_report(y_test, y_pred, target_names=["Non P4 D1", "P4"]))
print(confusion_matrix(y_test, y_pred))

auc = roc_auc_score(y_test, y_proba)
print(f"\nAUC-ROC: {auc:.4f}")

importances = pd.Series(xgb_p4.feature_importances_, index=FEATURES_P4).sort_values(ascending=False)
print(f"\nFeature importance:")
print(importances.to_string())

# Stratified 5-Fold CV
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv = cross_validate(
    xgb_p4, X, y, cv=skf,
    scoring=["roc_auc", "f1", "precision", "recall", "accuracy"],
    return_train_score=True,
)

print(f"\nStratified 5-Fold CV:")
print("=" * 60)
for metric in ["roc_auc", "accuracy", "f1", "precision", "recall"]:
    tr = cv[f"train_{metric}"]
    te = cv[f"test_{metric}"]
    label = "AUC-ROC" if metric == "roc_auc" else metric
    print(f"  {label:>12}:  train {tr.mean():.3f} (+/- {tr.std():.3f})  |  test {te.mean():.3f} (+/- {te.std():.3f})")
print(f"\n  Overfit gap (AUC): {cv['train_roc_auc'].mean() - cv['test_roc_auc'].mean():.3f}")

# Calibration
xgb_p4_cal = CalibratedClassifierCV(xgb_p4, method="isotonic", cv=5)
xgb_p4_cal.fit(X_train, y_train)
y_proba_cal = xgb_p4_cal.predict_proba(X_test)[:, 1]
y_pred_cal = (y_proba_cal >= 0.35).astype(int)

fraction_raw, mean_raw = calibration_curve(y_test, y_proba, n_bins=8)
fraction_cal, mean_cal = calibration_curve(y_test, y_proba_cal, n_bins=8)

brier_raw = brier_score_loss(y_test, y_proba)
brier_cal = brier_score_loss(y_test, y_proba_cal)

print(f"\nCalibrated model (threshold=0.35):")
print("=" * 60)
print(classification_report(y_test, y_pred_cal, target_names=["Non P4 D1", "P4"]))
print(confusion_matrix(y_test, y_pred_cal))

print(f"\nBrier:  raw = {brier_raw:.4f}  |  calibrated = {brier_cal:.4f}  |  improvement = {brier_raw - brier_cal:.4f}")

print(f"\nCalibration comparison (raw vs calibrated):")
print("=" * 75)
print(f"  {'--- RAW ---':^28}   |   {'--- CALIBRATED ---':^28}")
print(f"  {'Predicted':>10}  {'Actual':>8}  {'Gap':>8}   |   {'Predicted':>10}  {'Actual':>8}  {'Gap':>8}")
print(f"  {'-'*28}   |   {'-'*28}")

max_rows = max(len(mean_raw), len(mean_cal))
for i in range(max_rows):
    if i < len(mean_raw):
        r_gap = mean_raw[i] - fraction_raw[i]
        raw_str = f"  {mean_raw[i]:>10.2f}  {fraction_raw[i]:>8.2f}  {r_gap:>+8.2f}"
    else:
        raw_str = f"  {'':>28}"
    if i < len(mean_cal):
        c_gap = mean_cal[i] - fraction_cal[i]
        cal_str = f"{mean_cal[i]:>10.2f}  {fraction_cal[i]:>8.2f}  {c_gap:>+8.2f}"
        flag = "  <- off" if abs(c_gap) > 0.10 else ""
        cal_str += flag
    else:
        cal_str = f"{'':>28}"
    print(f"{raw_str}   |   {cal_str}")

auc_cal = roc_auc_score(y_test, y_proba_cal)
max_gap_cal = max(abs(mean_cal[i] - fraction_cal[i]) for i in range(len(mean_cal)))
print(f"\nAUC-ROC (calibrated): {auc_cal:.4f}")
print(f"Max calibration gap: {max_gap_cal:.2f}")

# Comparison
print(f"\n{'=' * 60}")
print("COMPARISON: LogReg baseline vs XGBoost + d1_prob")
print(f"{'=' * 60}")
rpt = classification_report(y_test, y_pred, target_names=["Non P4 D1", "P4"], output_dict=True)
print(f"  LogReg (8 feat):       P4 F1=TBD  acc=TBD")
print(f"  XGBoost (9 feat):      P4 F1={rpt['P4']['f1-score']:.2f}  acc={rpt['accuracy']:.2f}  AUC={auc:.3f}  Brier(cal)={brier_cal:.4f}")

Class balance: 1226 Non-P4 D1, 277 P4  (ratio 4.43:1)

XGBoost — 9 Features (8 core + d1_prob)
              precision    recall  f1-score   support

   Non P4 D1       0.88      0.70      0.78       246
          P4       0.30      0.58      0.40        55

    accuracy                           0.68       301
   macro avg       0.59      0.64      0.59       301
weighted avg       0.78      0.68      0.71       301

[[172  74]
 [ 23  32]]

AUC-ROC: 0.7164

Feature importance:
exit_velo_max    0.145806
pop_time         0.121882
d1_prob          0.120302
height           0.117440
weight           0.108057
c_velo           0.100439
sixty_time       0.096910
distance_max     0.095304
bat_speed_max    0.093860

Stratified 5-Fold CV:
       AUC-ROC:  train 0.897 (+/- 0.004)  |  test 0.654 (+/- 0.032)
      accuracy:  train 0.794 (+/- 0.015)  |  test 0.671 (+/- 0.021)
            f1:  train 0.600 (+/- 0.018)  |  test 0.349 (+/- 0.038)
     precision:  train 0.468 (+/- 0.021)  |  test 0.275 

In [20]:
import joblib, json, os
from datetime import datetime

# ============================================================
# SAVE CALIBRATED P4 MODEL — CATCHERS
# ============================================================
VERSION = f"version_{datetime.now().strftime('%m%d%Y')}"
SAVE_DIR = os.path.abspath(os.path.join(
    os.getcwd(), "..", "..", "..", "models", "models_c", "models_p4_or_not_c", VERSION
))
os.makedirs(SAVE_DIR, exist_ok=True)

THRESHOLD = 0.35

# 1. Save calibrated model
model_path = os.path.join(SAVE_DIR, "calibrated_xgb_model.pkl")
joblib.dump(xgb_p4_cal, model_path)
print(f"Saved calibrated model -> {model_path}")

# 2. Model config
model_config = {
    "model_type": "XGBClassifier + CalibratedClassifierCV (isotonic)",
    "task": "P4 vs Non-P4 D1 — Catchers",
    "version": VERSION,
    "created": datetime.now().isoformat(),
    "threshold": THRESHOLD,
    "n_features": len(FEATURES_P4),
    "features": FEATURES_P4,
    "hyperparameters": {
        "n_estimators": 300,
        "max_depth": 3,
        "learning_rate": 0.03,
        "subsample": 0.8,
        "colsample_bytree": 0.8,
        "reg_alpha": 1.5,
        "reg_lambda": 5.0,
        "min_child_weight": 8,
        "scale_pos_weight": round(neg / pos, 4),
    },
    "calibration": {
        "method": "isotonic",
        "cv": 5,
    },
    "metrics": {
        "holdout_auc_roc": round(auc_cal, 4),
        "holdout_brier_raw": round(brier_raw, 4),
        "holdout_brier_calibrated": round(brier_cal, 4),
        "max_calibration_gap": round(max_gap_cal, 4),
        "cv_test_auc_mean": round(cv["test_roc_auc"].mean(), 4),
        "cv_test_auc_std": round(cv["test_roc_auc"].std(), 4),
        "cv_train_auc_mean": round(cv["train_roc_auc"].mean(), 4),
        "overfit_gap_auc": round(cv["train_roc_auc"].mean() - cv["test_roc_auc"].mean(), 4),
    },
    "calibration_curve": {
        "predicted": [round(float(x), 4) for x in mean_cal],
        "actual": [round(float(x), 4) for x in fraction_cal],
    },
    "training_data": {
        "total_d1_players": int(neg + pos),
        "p4_players": int(pos),
        "non_p4_d1_players": int(neg),
        "stale_outliers_removed": 469,
        "stale_cleaning_passes": 5,
    },
}

config_path = os.path.join(SAVE_DIR, "model_config.json")
with open(config_path, "w") as f:
    json.dump(model_config, f, indent=2)
print(f"Saved model config -> {config_path}")

# 3. Feature metadata
feature_metadata = {
    "features": FEATURES_P4,
    "n_features": len(FEATURES_P4),
    "meta_features": {
        "d1_prob": {
            "source": "Out-of-fold predictions from catcher D1 model (CalibratedClassifierCV, isotonic, cv=3)",
            "generation": "cross_val_predict with StratifiedKFold(n_splits=5) on full catcher dataset (D1 + Non-D1)",
            "note": "Leak-free OOF probabilities — each player's d1_prob was predicted by a model that never saw that player in training",
        }
    },
    "feature_importance": {k: round(float(v), 4) for k, v in importances.items()},
    "position": "C",
    "core_features": ["height", "weight", "exit_velo_max", "distance_max", "sixty_time", "c_velo", "pop_time", "bat_speed_max"],
}

feat_path = os.path.join(SAVE_DIR, "feature_metadata.json")
with open(feat_path, "w") as f:
    json.dump(feature_metadata, f, indent=2)
print(f"Saved feature metadata -> {feat_path}")

# Summary
print(f"\n{'=' * 60}")
print(f"CATCHER P4 MODEL SAVED")
print(f"{'=' * 60}")
print(f"  Directory:    {SAVE_DIR}")
print(f"  Threshold:    {THRESHOLD}")
print(f"  Features:     {len(FEATURES_P4)} ({', '.join(FEATURES_P4)})")
print(f"  AUC-ROC:      {auc_cal:.4f}")
print(f"  Brier (cal):  {brier_cal:.4f}")
print(f"  Max cal gap:  {max_gap_cal:.2f}")
print(f"  Overfit gap:  {cv['train_roc_auc'].mean() - cv['test_roc_auc'].mean():.3f}")
print(f"\nFiles:")
for f in sorted(os.listdir(SAVE_DIR)):
    size = os.path.getsize(os.path.join(SAVE_DIR, f))
    print(f"  {f:>30}  ({size:,} bytes)")

Saved calibrated model -> /Users/ryankolodziejczyk/Documents/AI Baseball Recruitment/code/backend/ml/models/models_c/models_p4_or_not_c/version_04212026/calibrated_xgb_model.pkl
Saved model config -> /Users/ryankolodziejczyk/Documents/AI Baseball Recruitment/code/backend/ml/models/models_c/models_p4_or_not_c/version_04212026/model_config.json
Saved feature metadata -> /Users/ryankolodziejczyk/Documents/AI Baseball Recruitment/code/backend/ml/models/models_c/models_p4_or_not_c/version_04212026/feature_metadata.json

CATCHER P4 MODEL SAVED
  Directory:    /Users/ryankolodziejczyk/Documents/AI Baseball Recruitment/code/backend/ml/models/models_c/models_p4_or_not_c/version_04212026
  Threshold:    0.35
  Features:     9 (height, weight, exit_velo_max, distance_max, sixty_time, c_velo, pop_time, bat_speed_max, d1_prob)
  AUC-ROC:      0.7127
  Brier (cal):  0.1349
  Max cal gap:  0.16
  Overfit gap:  0.243

Files:
        calibrated_xgb_model.pkl  (1,948,148 bytes)
           feature_metada